Para estructurar esta sesión de la mejor manera, te presento un diseño de clase pedagógico de nivel intermedio, ideal para estudiantes de Ingeniería, Ciencia de Datos o Análisis de Negocios. 

El enfoque es **"aprender haciendo"** y utiliza la base de datos basada en el contexto de Caucasia para conectar la teoría con un problema corporativo real.



Al final del diseño encontrarás el **código completo y unificado** (la creación de la base de datos y la implementación de la clase en Python) listo para que tus alumnos lo ejecuten.



# Diseño de la Clase: Ingeniería de Datos con Python y SQL

* **Objetivo de la sesión:** Aprender a conectarse a bases de datos relacionales desde Python, diagnosticar la calidad de los datos (valores faltantes), realizar imputaciones estratégicas y generar reportes agregados utilizando agrupamientos corporativos (`GROUP BY`).



# Cronograma de la Sesión

## 1. Introducción y Conexión

* Explicar el patrón de arquitectura donde Python actúa como la "capa de aplicación" y SQLite como la "capa de persistencia".


* Discutir la importancia de no descargar tablas enteras a ciegas, sino interactuar con la base de datos mediante consultas eficientes.



## 2. Calidad de Datos y Tratamiento de Faltantes (40 min)

* **El problema:** En el mundo real, los clientes no siempre reportan sus ingresos o las encuestas de satisfacción quedan incompletas.


* **Toma de decisiones:** ¿Cuándo eliminar filas con datos faltantes (`NaN`) y cuándo imputarlos (llenarlos artificialmente)?


* **Técnica avanzada:** En lugar de rellenar los ingresos con un promedio global burdo, enseñaremos a los alumnos a **imputar por categorías** (ej. rellenar el ingreso faltante de un comerciante usando el promedio de los otros comerciantes del mismo sector).



## 3. Agrupamiento e Inteligencia de Negocios (40 min)

* Explicar el concepto de "Dividir - Aplicar - Combinar" (*Split-Apply-Combine*) de la operación `GROUP BY`.
* Mostrar cómo consolidar un millón de registros de transacciones en un pequeño reporte ejecutivo de métricas clave por sector económico.



## 4. Conclusiones y Reto de Cierre (20 min)

* Discutir el impacto del código limpio y modular (Programación Orientada a Objetos) frente a escribir scripts sueltos.



# Código Integrado

Este script hace dos cosas de forma automática:

1. **Fase 1:** Crea la base de datos SQLite (`caucasia_negocios.db`) inyectando intencionalmente registros con **datos faltantes (None / NaN)** en ingresos y satisfacción para que sirva como laboratorio de pruebas.


2. **Fase 2:** Implementa la clase `AnalizadorDatosCaucasia` que encapsula todo el ciclo de vida de los datos mediante métodos limpios y profesionales.


In [1]:
import sqlite3
import pandas as pd
import numpy as np


In [5]:

# =====================================================================
# FASE 1: SCRIPT DE CREACIÓN DE LA BASE DE DATOS (Laboratorio con NaNs)
# =====================================================================
def inicializar_base_de_datos():
    """Crea la base de datos y simula datos reales con valores faltantes."""
    conexion = sqlite3.connect('caucasia_negocios.db')
    cursor = conexion.cursor()
    
    # Crear tablas
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS clientes (
        identificador_cliente INTEGER PRIMARY KEY,
        sector_economico TEXT
    );
    """)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS ingresos_credito (
        identificador_cliente INTEGER,
        ingresos_mensuales_cop REAL,
        FOREIGN KEY(identificador_cliente) REFERENCES clientes(identificador_cliente)
    );
    """)
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS satisfaccion_operaciones (
        identificador_cliente INTEGER,
        puntuacion_satisfaccion REAL
    );
    """)
    
    # Insertar Datos de Clientes
    sectores = ['Comercio', 'Agropecuario', 'Minería', 'Servicios']
    datos_clientes = [(i, sectores[i % 4]) for i in range(1, 101)] # 100 clientes
    cursor.executemany("INSERT OR REPLACE INTO clientes VALUES (?, ?);", datos_clientes)
    
    # Insertar Ingresos (Simulando un 15% de datos faltantes 'None')
    datos_ingresos = []
    for i in range(1, 101):
        if i % 7 == 0: # Cada 7 registros, el dato se pierde
            ingreso = None
        else:
            # Ingreso base según el sector para que la imputación tenga sentido lógico
            base = 2000000 if i % 4 == 0 else 4500000
            ingreso = base + (i * 15000)
        datos_ingresos.append((i, ingreso))
    cursor.executemany("INSERT OR REPLACE INTO ingresos_credito VALUES (?, ?);", datos_ingresos)
    
    # Insertar Historial de Satisfacción (1000 registros, algunos sin calificación)
    datos_satisfaccion = []
    for i in range(1, 1001):
        cliente_id = (i % 100) + 1
        puntuacion = None if i % 13 == 0 else float(np.random.randint(1, 6))
        datos_satisfaccion.append((cliente_id, puntuacion))
    cursor.executemany("INSERT INTO satisfaccion_operaciones VALUES (?, ?);", datos_satisfaccion)
    
    conexion.commit()
    conexion.close()
    print("Base de datos 'caucasia_negocios.db' creada con éxito (incluye datos faltantes para el taller).")

# Executar inicialización
inicializar_base_de_datos()



Base de datos 'caucasia_negocios.db' creada con éxito (incluye datos faltantes para el taller).


In [ ]:

# =====================================================================
# FASE 2: DISEÑO DE LA CLASE DE PROCESAMIENTO Y ANÁLISIS
# =====================================================================
class AnalizadorDatosCaucasia:
    """Clase empresarial para gestionar el acceso, limpieza y agregación 
       de la información del ecosistema de negocios de Caucasia."""
    
    def __init__(self, ruta_db):
        """Establece la ruta de la base de datos."""
        self.ruta_db = ruta_db
        self.df_consolidado = None

    def extraer_datos_completos(self):
        """
        Paso 1: Tarea de Acceso a Base de Datos.
        Realiza un JOIN en el motor SQL para traer la información base a Python.
        """
        conexion = sqlite3.connect(self.ruta_db)
        
        consulta = """
            SELECT c.identificador_cliente, c.sector_economico, 
                   i.ingresos_mensuales_cop, s.puntuacion_satisfaccion
            FROM clientes c
            LEFT JOIN ingresos_credito i ON c.identificador_cliente = i.identificador_cliente
            LEFT JOIN satisfaccion_operaciones s ON c.identificador_cliente = s.identificador_cliente
        """
        # Explicar a los alumnos el uso de LEFT JOIN para no perder clientes que tengan nulos
        self.df_consolidado = pd.read_sql_query(consulta, conexion)
        conexion.close()
        print(f" Datos extraídos con éxito. Filas totales: {len(self.df_consolidado)}")
        return self.df_consolidated_preview()

    def diagnosticar_datos_faltantes(self):
        """
        Paso 2: Auditoría de Calidad de Datos.
        Muestra cuántos valores nulos existen por columna.
        """
        if self.df_consolidado is None:
            raise ValueError("Primero debes extraer los datos con 'extraer_datos_completos()'")
        
        print("\n --- DIAGNÓSTICO DE VALORES FALTANTES ---")
        nulos_por_columna = self.df_consolidado.isnull().sum()
        porcentaje_nulos = (self.df_consolidado.isnull().sum() / len(self.df_consolidado)) * 100
        
        diagnostico = pd.DataFrame({
            'Valores Faltantes (Cant)': nulos_por_columna,
            'Porcentaje (%)': porcentaje_nulos
        })
        return diagnostico

    def tratar_datos_faltantes(self):
        """
        Paso 3: Tratamiento e Imputación Avanzada.
        - Satisfacción: Se eliminan filas nulas (política de la empresa: no inventar calificaciones).
        - Ingresos: Se imputan usando la media ponderada DEL SECTOR ECONÓMICO al que pertenecen.
        """
        if self.df_consolidado is None:
            raise ValueError("No hay datos cargados para limpiar.")

        print("\n --- INICIANDO PROCESO DE LIMPIEZA E IMPUTACIÓN ---")
        
        # 1. Eliminación dirigida (Estrategia Dropna)
        antes_drop = len(self.df_consolidado)
        self.df_consolidado.dropna(subset=['puntuacion_satisfaccion'], inplace=True)
        print(f"   -> Eliminadas {antes_drop - len(self.df_consolidado)} filas con calificación de satisfacción vacía.")
        
        # 2. Imputación inteligente por grupo (Estrategia Groupby + Transform)
        # Calculamos la media de ingresos por sector y la usamos para rellenar los NaNs del mismo sector
        self.df_consolidado['ingresos_mensuales_cop'] = self.df_consolidado.groupby('sector_economico')['ingresos_mensuales_cop']\
                                                            .transform(lambda x: x.fillna(x.mean()))
        
        print("   -> Ingresos faltantes imputados con éxito utilizando la media de su respectivo Sector Económico.")
        return self.df_consolidado.isnull().sum()

    def generar_reporte_ejecutivo(self):
        """
        Paso 4: Agrupamiento estratégico (GROUPBY).
        Agrupa la información limpia para calcular métricas agregadas por Sector Financiero.
        """
        if self.df_consolidado is None:
            raise ValueError("No hay datos limpios para agrupar.")
            
        print("\n --- GENERANDO REPORTE AGREGADO EJECUTIVO ---")
        
        # Agrupamos por sector económico y calculamos múltiples métricas a la vez
        reporte = self.df_consolidado.groupby('sector_economico').agg(
            numero_clientes=('identificador_cliente', 'nunique'),
            volumen_operaciones=('puntuacion_satisfaccion', 'count'),
            ingreso_promedio_cop=('ingresos_mensuales_cop', 'mean'),
            satisfaccion_promedio=('puntuacion_satisfaccion', 'mean')
        ).reset_index()
        
        # Dar formato estético al reporte final
        reporte['ingreso_promedio_cop'] = reporte['ingreso_promedio_cop'].map('${:,.2f}'.format)
        reporte['satisfaccion_promedio'] = reporte['satisfaccion_promedio'].map('{:.2f} / 5.0'.format)
        
        return reporte

    def df_consolidated_preview(self):
        return self.df_consolidado.head(5)


In [7]:


# =====================================================================
# FASE 3: DEMOSTRACIÓN PRÁCTICA EN VIVO (Simulación de la Clase)
# =====================================================================
if __name__ == "__main__":
    print("\n--- INICIO DE LA CLASE EN VIVO ---")
    
    # 1. Instanciar la clase asignándole la base de datos del caso
    analizador = AnalizadorDatosCaucasia(ruta_db='caucasia_negocios.db')
    
    # 2. Extracción (Acceso)
    print("\n[Paso 1: Extracción de Datos]")
    print(analizador.extraer_datos_completos())
    
    # 3. Diagnóstico de Nulos
    print(analizador.diagnosticar_datos_faltantes())
    
    # 4. Tratamiento e Imputación
    analizador.tratar_datos_faltantes()
    
    # Re-verificar que ya no queden nulos
    print("\nVerificación post-limpieza (Debe salir 0 nulos):")
    print(analizador.df_consolidado.isnull().sum())
    
    # 5. Agrupamiento e Inteligencia de Negocios
    reporte_final = analizador.generar_reporte_ejecutivo()
    print("\n REPORTE FINAL ENTREGADO AL COMITÉ DIRECTIVO:")
    print(reporte_final.to_string(index=False))



--- INICIO DE LA CLASE EN VIVO ---

[Paso 1: Extracción de Datos]
📥 Datos extraídos con éxito. Filas totales: 4000
   identificador_cliente sector_economico  ingresos_mensuales_cop  \
0                      1     Agropecuario               4515000.0   
1                      1     Agropecuario               4515000.0   
2                      1     Agropecuario               4515000.0   
3                      1     Agropecuario               4515000.0   
4                      1     Agropecuario               4515000.0   

   puntuacion_satisfaccion  
0                      1.0  
1                      1.0  
2                      1.0  
3                      1.0  
4                      1.0  

🔍 --- DIAGNÓSTICO DE VALORES FALTANTES ---
                         Valores Faltantes (Cant)  Porcentaje (%)
identificador_cliente                           0             0.0
sector_economico                                0             0.0
ingresos_mensuales_cop                        560    


# Guía didáctica al mostrar el código:

1. **La magia de `.transform()`:** Explícales que la línea `groupby(...).transform(lambda x: x.fillna(x.mean()))` es una joya de ingeniería en Pandas. A diferencia de un `groupby` normal que reduce el tamaño del dataset, `.transform()` calcula la media del subgrupo, pero mantiene la estructura original para rellenar el dato vacío exactamente donde hacía falta.


2. **Buenas Prácticas:** Haz énfasis en que la clase separa las responsabilidades. Si el día de mañana la base de datos cambia de SQLite a PostgreSQL, solo se modifica la línea de conexión dentro del método `extraer_datos_completos`, mientras que toda la lógica de limpieza e imputación de Pandas seguirá funcionando intacta.